In [ ]:
!pip install -r requirements.txt

## Run measurements

Before running this, make sure the model files are uploaded in the `models` folder and test images (.jpg) are uploaded in the `data\test\image` folder.

Select `test_bg_change.py` for landmark detection with clothing background switched to green (which was shown to imporve detection accuracy), or choose `test_no_bg_change.py` to do without background change. 

Flag `do_refine` selects whether to refine landmark positions after detection, which can offer moderate improvements in cases. 

The results will be saved under the folder `output` with a time tag. Inside each time-tagged folder we have individual folders containing each of the garment classes. Inside each class folder we have 3 folders: 

    images : containing the output images with detected landmarks 
    coordinates : contains the measurements json files (e.g. meas_vest_1.jpg.json); and coordinates json files 
    (e.g. coords_vest_1.jpg.json) for each image. These will be used later for evaluation
    segmentation : contains the segmentation masks for the images



In [ ]:
!python tools/test_bg_change.py \ 
  --cfg experiments/deepfashion2/hrnet/w48_384x288_adam_lr1e-3_zara.yaml \
  --model_file models/pose_hrnet-w48_384x288-deepfashion2_mAP_0.7017.pth \
  --cls_model_file models/tiny_vit_inditex_finetuned.pt \
  --data_dir data \
  --do_refine True \

^C


## Build the csv file from the mannually labeled test data

This will use our manually labeled test dataset in the "test_self_annotated" folder and generate a computed_aspects.csv file containing the aspect ratios in a format specified by Inditex.

In [2]:
import os, glob, json
import pandas as pd

# ────────────────────────────────────────────────────────────────────────────────
# CONFIG: where your per‐class folders live & where to dump the CSV
# ────────────────────────────────────────────────────────────────────────────────
BASE_DIR = "test_self_annotated"
OUT_CSV  = "computed_aspects.csv"

# exact header & order you asked for
COLUMNS = [
    "ID_ARTICLE",
    "DESC_SECTION",
    "PICTURES",
    "CHEST_OVER_FRONT_LENGTH",
    "SLEEVE_LENGTH_OVER_FRONT_LENGTH",
    "BACK_WIDTH_OVER_ARM_WIDTH",
    "HIP_OVER_WAIST",
    "FRONT_LENGTH_OVER_CHEST",
    "BACK_RISE_OVER_FRONT_RISE",
]

# for each of those 6 aspect‐ratios, which two absolute measurements form the numerator and denominator?
ASPECT_DEFS = {
    "CHEST_OVER_FRONT_LENGTH":           ("chest",         "front length"),
    "SLEEVE_LENGTH_OVER_FRONT_LENGTH":  ("sleeve length", "front length"),
    "BACK_WIDTH_OVER_ARM_WIDTH":        ("back width",    "sleeve width"),
    "HIP_OVER_WAIST":                   ("hips",          "waist"),
    "FRONT_LENGTH_OVER_CHEST":          ("front length",  "chest"),
    # BACK_RISE_OVER_FRONT_RISE we’ll leave blank
}

# these are *all* of the synonyms you might have used
# (pulled from your earlier csv_to_aspect_keys for aspect‐ratios)
CSV_SYNONYMS = {
    "CHEST_OVER_FRONT_LENGTH": [
        "chest/front length",
        "chest/full length",
    ],
    "SLEEVE_LENGTH_OVER_FRONT_LENGTH": [
        "sleeve length/front length",
        "sleeve length/full length",
    ],
    "BACK_WIDTH_OVER_ARM_WIDTH": [
        "back width/sleeve width",
    ],
    "HIP_OVER_WAIST": [
        "hips/waist",
    ],
    "FRONT_LENGTH_OVER_CHEST": [
        "front length/chest",
        "full length/chest",
    ],
}


# ────────────────────────────────────────────────────────────────────────────────
# Build a reverse‐map: any synonym → the single canonical ABSOLUTE label
# ────────────────────────────────────────────────────────────────────────────────
synonyms_map = {}
for col, (num_lbl, den_lbl) in ASPECT_DEFS.items():
    for syn in CSV_SYNONYMS.get(col, []):
        num_syn, den_syn = [s.strip() for s in syn.split("/")]
        synonyms_map[num_syn] = num_lbl
        synonyms_map[den_syn] = den_lbl

# also map each canonical back to itself
for num_lbl, den_lbl in ASPECT_DEFS.values():
    synonyms_map[num_lbl] = num_lbl
    synonyms_map[den_lbl] = den_lbl


# ────────────────────────────────────────────────────────────────────────────────
# Helpers
# ────────────────────────────────────────────────────────────────────────────────
def get_dist(data, a, b):
    """Find a→b or b→a in data[...]['distances'] or return None."""
    a, b = str(a), str(b)
    if a in data and b in data[a].get("distances", {}):
        return data[a]["distances"][b]
    if b in data and a in data[b].get("distances", {}):
        return data[b]["distances"][a]
    return None

def safe_ratio(numer, denom):
    """Return numer/denom or blank if missing/zero."""
    if numer is None or denom is None or denom == 0:
        return ""
    return numer / denom


# ────────────────────────────────────────────────────────────────────────────────
# MAIN
# ────────────────────────────────────────────────────────────────────────────────
records = []

for test_folder in os.listdir(BASE_DIR):
    if not test_folder.startswith("test_"):
        continue

    class_dir = os.path.join(BASE_DIR, test_folder)
    json_dir  = os.path.join(class_dir, "json")
    dict_path = os.path.join(class_dir, "dict.json")

    if not os.path.isdir(json_dir) or not os.path.isfile(dict_path):
        print(f"Skipping `{test_folder}` (missing json/ or dict.json)")
        continue

    # load dict.json → maybe nested under { "<class>": { ... } }
    mapping = json.load(open(dict_path))
    cls = test_folder[len("test_"):]
    if cls in mapping:
        mapping = mapping[cls]

    # invert into { canonical_label: (k1,k2) }
    absolutes_map = {}
    for pair_str, meas_label in mapping.items():
        # canonicalize the meas_label via synonyms_map
        canon = synonyms_map.get(meas_label, meas_label)
        k1, k2 = [s.strip() for s in pair_str.split("<->")]
        absolutes_map[canon] = (k1, k2)

    # now scan each image‐json
    for jpath in glob.glob(os.path.join(json_dir, "*.json")):
        art_id = os.path.splitext(os.path.basename(jpath))[0]
        data   = json.load(open(jpath, "r"))

        # pull out all absolutes we can
        absolutes = {
            label: get_dist(data, *pair)
            for label, pair in absolutes_map.items()
        }

        # build one row
        row = {
            "ID_ARTICLE":    art_id,
            "DESC_SECTION":  "",
            "PICTURES":      "[]",
        }

        for col, (num_lbl, den_lbl) in ASPECT_DEFS.items():
            row[col] = safe_ratio(
                absolutes.get(num_lbl),
                absolutes.get(den_lbl)
            )

        # skip back‐rise/front‐rise
        row["BACK_RISE_OVER_FRONT_RISE"] = ""

        records.append(row)

# dump to DataFrame → CSV
df = pd.DataFrame(records, columns=COLUMNS)
df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(df)} rows to {OUT_CSV}")
df.head()


Wrote 90 rows to computed_aspects.csv


,ID_ARTICLE,DESC_SECTION,PICTURES,CHEST_OVER_FRONT_LENGTH,SLEEVE_LENGTH_OVER_FRONT_LENGTH,BACK_WIDTH_OVER_ARM_WIDTH,HIP_OVER_WAIST,FRONT_LENGTH_OVER_CHEST,BACK_RISE_OVER_FRONT_RISE
0,long_sleeve_dress_1,,[],0.233067,0.450648,,1.476421,4.290604,
1,long_sleeve_dress_10,,[],0.223231,0.49639,,1.181853,4.479674,
2,long_sleeve_dress_2,,[],0.260839,0.578504,,1.422223,3.833775,
3,long_sleeve_dress_3,,[],0.217111,0.449011,,1.430422,4.60593,
4,long_sleeve_dress_4,,[],0.33182,0.605316,,1.324174,3.013684,


## Compare measurements to ground truth

This will compute the mean fractional errors in the derived aspect ratios compared to ground truths.  

To change the output directory, please edit the output directory in this line `pattern = f"output/**/{time tag to be changed}/class_*/coordinates/meas_{art_id}.jpg.json"`.

In [ ]:
# Use mean fractional error in the aspect ratios per class per measurement
import os
import glob
import json

import pandas as pd
import numpy as np

# 1) Load your CSV
df = pd.read_csv("computed_aspects.csv") # use "measures_data.csv" for the original inditex file
                                                # use "computed_aspects.csv" for the self-annoated data
# 2) classifier→detector map and labels
CLS_TO_DET = {
    0:11, 1:2,  2:10, 3:1,
    4:7,  5:9,  6:8,  7:5,
    8:12
}
CLOTHING_LABELS = {
    "0": "long sleeve dress",
    "1": "long sleeve top",
    "2": "short sleeve dress",
    "3": "short sleeve top",
    "4": "shorts",
    "5": "skirt",
    "6": "trousers",
    "7": "vest",
    "8": "vest dress"
}

# 3) Map CSV columns → list of possible JSON aspect_ratio keys
csv_to_aspect_keys = {
    "CHEST_OVER_FRONT_LENGTH": [
        "chest/front length",
        "chest/full length"
    ],
    "SLEEVE_LENGTH_OVER_FRONT_LENGTH": [
        "sleeve length/front length",
        "sleeve length/full length"
    ],
    "BACK_WIDTH_OVER_ARM_WIDTH": [
        "back width/sleeve width"
    ],
    "HIP_OVER_WAIST": [
        "hips/waist",
        "hips/wasit"
    ],
    "FRONT_LENGTH_OVER_CHEST": [
        "front length/chest",
        "full length/chest"
    ],
    # we skip BACK_RISE_OVER_FRONT_RISE for now
}

# 4) Collect fractional‐error records
records = []
for _, row in df.iterrows():
    art_id = str(row["ID_ARTICLE"])
    # adjust this pattern to wherever your predictions live:
    # pattern = f"output/**/class_*/coordinates/meas_{art_id}_1.jpg.json"
    pattern = f"output/**/2025-05-22-19-31/class_*/coordinates/meas_{art_id}.jpg.json"
    paths = glob.glob(pattern, recursive=True)
    if not paths:
        continue

    meas         = json.load(open(paths[0], "r"))
    pred_aspects = meas.get("aspect_ratio", {})
    cat_id       = meas.get("category_id")

    for csv_col, aspect_keys in csv_to_aspect_keys.items():
        gt = row.get(csv_col)
        if pd.isna(gt) or float(gt) == 0:
            continue
        gt = float(gt)

        # find the first matching predicted key
        pred = None
        for key in aspect_keys:
            if key in pred_aspects:
                pred = pred_aspects[key]
                break
        if pred is None:
            continue

        frac_err = abs(pred - gt) / gt
        records.append({
            "article_id":    art_id,
            "category_id":   cat_id,
            "measurement":   csv_col,
            "gt":            gt,
            "pred":          pred,
            "fractional_error": frac_err
        })

# 5) Build DataFrame
res = pd.DataFrame(records)
if res.empty:
    print("No matching predictions + ground‐truth pairs found.")
    exit()

# 6) Compute mean fractional error per (measurement, category_id)
mean_frac_pc = (
    res
    .groupby(["measurement","category_id"])["fractional_error"]
    .mean()
    .unstack(fill_value=np.nan)
)

# 7) Map detector category_id back to human‐readable clothing labels
det_to_clsidx = {v:k for k,v in CLS_TO_DET.items()}
det_to_name   = {
    det_id: CLOTHING_LABELS[str(cls_idx)]
    for det_id, cls_idx in det_to_clsidx.items()
}
mean_frac_pc_named = mean_frac_pc.rename(columns=det_to_name)

print("\n=== Mean fractional error per measurement per clothing type ===")
print(mean_frac_pc_named)

# 8) (Optional) save to CSV
mean_frac_pc_named.to_csv("mean_fractional_error_by_class_no_bg_change.csv")
